# Building World-Aware Agents with the OpenAI Agents SDK

LLMs have a knowledge cutoff. Ask your agent "what's the probability of a US recession?" and it will either hallucinate a number, hedge, or give you stale data.

This cookbook shows how to build agents that have **real-time world awareness** — agents that know today's geopolitical risk level, current oil prices, recession probabilities, and Fed rate expectations. Not from web search (which returns narratives), but from **calibrated probability data** backed by real money.

We use [SimpleFunctions](https://simplefunctions.dev/world), which aggregates 9,706 prediction market contracts into an ~800-token world state snapshot. Prediction markets produce calibrated probabilities because participants lose money when they're wrong — a fundamentally different incentive than news, polls, or LLM reasoning.

**What you'll build:**
1. An agent with dynamic world-aware instructions
2. Tools for focused topic drilling and incremental updates
3. A multi-agent system where all agents share the same world context

## Setup

Install dependencies and set your OpenAI API key.

In [ ]:
%pip install openai-agents requests -q

In [ ]:
import os
import requests

# No API key needed for the world state endpoint — it's free and public.
# You only need your OpenAI key for the agent itself.
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"

## Step 1: Fetch the World State

One API call returns ~800 tokens of markdown covering geopolitics, economy, energy, elections, crypto, and tech. The data comes from 9,706 prediction market contracts on Kalshi (CFTC-regulated) and Polymarket.

Anchor contracts — recession probability, Fed rate path, Iran invasion — always appear regardless of daily noise. Other contracts are ranked by volume × price movement, not raw volatility.

In [ ]:
world = requests.get("https://simplefunctions.dev/api/agent/world").text
print(world)

Notice the format: every line is a fact with a number attached. `Iran invasion: 53c (+5c)` means the market prices Iran invasion at 53% probability, up 5 percentage points in 24 hours, with real money behind it. This is what calibrated data looks like — not "tensions remain elevated."

## Step 2: World-Aware Agent with Dynamic Instructions

The Agents SDK's `instructions` parameter accepts a callable. We use this to fetch fresh world state before each agent run.

In [ ]:
from agents import Agent, Runner


def get_instructions():
    world = requests.get("https://simplefunctions.dev/api/agent/world").text
    return f"""You are a macro research analyst with real-time world awareness.

Use the data below as ground truth. Cite specific probabilities and prices.
Do not hallucinate numbers — if it's not in the data, say you don't know.

{world}"""


analyst = Agent(
    name="world_aware_analyst",
    instructions=get_instructions,
)

result = Runner.run_sync(analyst, "What's the current geopolitical risk level and how does it affect energy markets?")
print(result.final_output)

The agent now cites real numbers — not because we told it to make up numbers, but because it has actual data to cite.

## Step 3: Adding Depth Tools

The world state snapshot is a panoramic view. For specific questions, the agent needs tools to drill deeper:

- **`world_state(focus=...)`** — same token budget, concentrated on fewer topics (e.g., 10 energy contracts instead of 4)
- **`world_delta(since=...)`** — only what changed since a timestamp (~30-50 tokens vs 800)
- **`search_markets(query=...)`** — find specific prediction market contracts

In [ ]:
import json
from agents import function_tool


@function_tool
def world_state(focus: str = "") -> str:
    """Get current world state from prediction markets.

    Args:
        focus: Comma-separated topics for deeper coverage.
               Options: geopolitics, economy, energy, elections, crypto, tech.
               Empty string returns all topics.
    """
    url = "https://simplefunctions.dev/api/agent/world"
    if focus:
        url += f"?focus={focus}"
    return requests.get(url).text


@function_tool
def world_delta(since: str = "1h") -> str:
    """Get only what changed in the world since the given time.
    Much smaller than full state — use for periodic refresh during long tasks.

    Args:
        since: Time window. Options: 30m, 1h, 6h, 24h.
    """
    return requests.get(
        f"https://simplefunctions.dev/api/agent/world/delta?since={since}"
    ).text


@function_tool
def search_markets(query: str) -> str:
    """Search prediction markets for specific contracts.
    Returns prices, volumes, and spreads from Kalshi and Polymarket.

    Args:
        query: Natural language search query, e.g. 'iran oil' or 'fed rate cut'.
    """
    resp = requests.get(
        f"https://simplefunctions.dev/api/public/scan?q={query}"
    )
    return json.dumps(resp.json(), indent=2)


analyst_with_tools = Agent(
    name="deep_analyst",
    instructions=get_instructions,
    tools=[world_state, world_delta, search_markets],
)

result = Runner.run_sync(
    analyst_with_tools,
    "I need a deep dive on Iran risk and its impact on oil. Search for specific contracts."
)
print(result.final_output)

## Step 4: Multi-Agent with Shared World Context

In multi-agent systems, consistency matters. If the researcher says "recession at 33%" and the risk analyst says "recession at 40%," they're not disagreeing — they're hallucinating different numbers.

The fix: fetch the world state once, share it across all agents.

In [ ]:
# Fetch once, share across all agents
shared_world = requests.get("https://simplefunctions.dev/api/agent/world").text

SHARED_CONTEXT = f"""You have real-time world awareness from prediction markets.
Cite these numbers directly. Do not make up probabilities.

{shared_world}"""

risk_analyst = Agent(
    name="risk_analyst",
    instructions=SHARED_CONTEXT + "\nYou are a risk analyst. Assess portfolio implications of the data. Never say 'could' when you have a specific probability.",
    tools=[world_state],
)

researcher = Agent(
    name="researcher",
    instructions=SHARED_CONTEXT + "\nYou are a macro researcher. Analyze geopolitical and economic developments. Hand off to the risk analyst when you identify concerning data.",
    handoffs=[risk_analyst],
    tools=[world_state, search_markets],
)

result = Runner.run_sync(
    researcher,
    "What should a portfolio manager know about geopolitical risk today?"
)
print(result.final_output)

Both agents see the same Iran invasion probability, the same recession odds, the same oil price. When the researcher identifies a risk and hands off to the risk analyst, the numbers are consistent.

## Why Prediction Markets Instead of Web Search?

| | Web Search | News API | Prediction Markets |
|---|---|---|---|
| **Output** | Narrative text | Headlines | Calibrated probabilities |
| **Token cost** | 2,000-5,000 | 500-1,000 | ~800 for everything |
| **Latency** | 2-5 seconds | 500ms | ~200ms (cached) |
| **Signal** | Needs parsing | No judgment | Numbers with money behind them |
| **Calibration** | None | None | Participants lose money when wrong |

A prediction market price of 53 cents on "Iran invasion" encodes the aggregate judgment of everyone with money at risk on that question. A news headline encodes what an editor thought would get clicks. These are fundamentally different information sources.

## Summary

| Pattern | When to use | Token cost |
|---|---|---|
| `instructions=get_instructions` | Agent needs fresh world state each run | ~800 |
| `world_state(focus="energy,geo")` | Agent needs deeper coverage of specific topics | ~800 (concentrated) |
| `world_delta(since="1h")` | Long-running agent needs periodic refresh | ~30-50 |
| `search_markets(query)` | Agent needs specific contract data | Varies |
| Shared `SHARED_CONTEXT` | Multi-agent consistency | ~800 (shared) |

The world state endpoint requires no API key, is free, and returns data from 9,706 prediction markets updated every 15 minutes.

**Links:**
- [World State API](https://simplefunctions.dev/api/agent/world) — try it now
- [Documentation](https://simplefunctions.dev/docs/guide)
- [All endpoints](https://simplefunctions.dev/api/tools)